In [1]:
import json

from torch.fx.experimental.symbolic_shapes import create_contiguous

with open("/Users/nad/mobiraph/data/n13_repbase_processed/hierarchy_sequences_02_ltr_correction_with_classes.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

print(type(metadata))

<class 'dict'>


In [2]:
general_dict = {}

In [3]:
for class_name, class_info in metadata.items():
    sequences = class_info["sequences"]
    for sequence in sequences:
        general_dict[sequence] = {"class": class_name}

In [4]:
for class_name, class_info in metadata.items():
    if class_name == 'Class II (DNA transposons)':
        sequences = class_info["sequences"]
        for sequence in sequences:
            general_dict[sequence]["order"] = "DNA transposon"

In [5]:
general_dict['MARINER62_CB']

{'class': 'Class II (DNA transposons)', 'order': 'DNA transposon'}

In [6]:
for supfam_name, supfam_info in metadata['Class II (DNA transposons)']["subs"].items():
    sequences = supfam_info["sequences"]
    for sequence in sequences:
        general_dict[sequence]["superfamily"] = supfam_name

In [7]:
for order_name, order_info in metadata['Class I (Retrotransposons)']["subs"].items():
    sequences = order_info["sequences"]
    for sequence in sequences:
        general_dict[sequence]["order"] = order_name

In [8]:
general_dict['LTR4_CR-LTR']

{'class': 'Class I (Retrotransposons)', 'order': 'LTR Retrotransposon'}

In [9]:
for supfam_name, supfam_info in metadata['Class I (Retrotransposons)']["subs"]["LTR Retrotransposon"]["subs"].items():
    sequences = supfam_info["sequences"]
    for sequence in sequences:
        general_dict[sequence]["superfamily"] = supfam_name

In [10]:
general_dict['GYPSY1-LTR_CB']

{'class': 'Class I (Retrotransposons)',
 'order': 'LTR Retrotransposon',
 'superfamily': 'Gypsy'}

In [11]:
for supfam_name, supfam_info in metadata['Class I (Retrotransposons)']["subs"]["Non-LTR Retrotransposon"]["subs"].items():
    sequences = supfam_info["sequences"]
    for sequence in sequences:
        general_dict[sequence]["superfamily"] = supfam_name

In [12]:
general_dict['SINEX-1_CR']

{'class': 'Class I (Retrotransposons)',
 'order': 'Non-LTR Retrotransposon',
 'superfamily': 'SINE'}

In [13]:
with open("/Users/nad/mobiraph/data/n13_repbase_processed/metadata_03.json", "w", encoding="utf-8") as f:
    json.dump(general_dict, f, ensure_ascii=False, indent=4)

In [14]:
sv_plants_category = {}

with open("/Users/nad/mobiraph/data/plant_sv_fam_orf_on_repbase_best.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if parts[1] not in general_dict:
            continue
        sv_plants_category[parts[0]] = general_dict[parts[1]]

In [15]:
with open("/Users/nad/mobiraph/data/n26_sv_processed/sv_plants_category.json", "w", encoding="utf-8") as f:
    json.dump(sv_plants_category, f, ensure_ascii=False, indent=4)

In [16]:
len(sv_plants_category)

11914

In [17]:
sv_insects_category = {}

with open("/Users/nad/mobiraph/data/insect_sv_fam_orf_on_repbase_best.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if parts[1] not in general_dict:
            continue
        sv_insects_category[parts[0]] = general_dict[parts[1]]

In [18]:
with open("/Users/nad/mobiraph/data/n26_sv_processed/sv_insects_category.json", "w", encoding="utf-8") as f:
    json.dump(sv_insects_category, f, ensure_ascii=False, indent=4)

In [19]:
len(sv_insects_category)

7773

In [20]:
def create_hierarchy_dict(g_dict):
    hierarchy_sequences_sv = {}
    # class level
    hierarchy_sequences_sv['Class I (Retrotransposons)'] = {'sequences': [], 'subs': {}}
    hierarchy_sequences_sv['Class II (DNA transposons)'] = {'sequences': [], 'subs': {}}
    for name, info in g_dict.items():
        hierarchy_sequences_sv[info['class']]['sequences'].append(name)
    # order level
    hierarchy_sequences_sv['Class I (Retrotransposons)']['subs']['LTR Retrotransposon'] = {'sequences': [], 'subs': {}}
    hierarchy_sequences_sv['Class I (Retrotransposons)']['subs']['Non-LTR Retrotransposon'] = {'sequences': [], 'subs': {}}
    for name, info in g_dict.items():
        if not info['order']:
            continue
        if info['class'] == 'Class I (Retrotransposons)':
            hierarchy_sequences_sv['Class I (Retrotransposons)']['subs'][info['order']]['sequences'].append(name)
    # superfamily level
    for superfamily in metadata['Class I (Retrotransposons)']['subs']['LTR Retrotransposon']['subs'].keys():
        hierarchy_sequences_sv['Class I (Retrotransposons)']['subs']['LTR Retrotransposon']['subs'][superfamily] = {'sequences': [], 'subs': {}}
    for superfamily in metadata['Class I (Retrotransposons)']['subs']['Non-LTR Retrotransposon']['subs'].keys():
        hierarchy_sequences_sv['Class I (Retrotransposons)']['subs']['Non-LTR Retrotransposon']['subs'][superfamily] = {'sequences': [], 'subs': {}}
    for superfamily in metadata['Class II (DNA transposons)']['subs'].keys():
        hierarchy_sequences_sv['Class II (DNA transposons)']['subs'][superfamily] = {'sequences': [], 'subs': {}}

    for name, info in g_dict.items():
        if 'superfamily' not in info:
            continue
        if info['class'] == 'Class I (Retrotransposons)':
            hierarchy_sequences_sv['Class I (Retrotransposons)']['subs'][info['order']]['subs'][info['superfamily']]['sequences'].append(name)
        if info['class'] == 'Class II (DNA transposons)':
            hierarchy_sequences_sv['Class II (DNA transposons)']['subs'][info['superfamily']]['sequences'].append(name)
    return hierarchy_sequences_sv


In [21]:
hierarchy_sequences_sv_plants = create_hierarchy_dict(sv_plants_category)
hierarchy_sequences_sv_insects = create_hierarchy_dict(sv_insects_category)

In [22]:
with open("/Users/nad/mobiraph/data/n26_sv_processed/hierarchy_sequences_sv_plants.json", "w", encoding="utf-8") as f:
    json.dump(hierarchy_sequences_sv_plants, f, ensure_ascii=False, indent=4)
with open("/Users/nad/mobiraph/data/n26_sv_processed/hierarchy_sequences_sv_insects.json", "w", encoding="utf-8") as f:
    json.dump(hierarchy_sequences_sv_insects, f, ensure_ascii=False, indent=4)

In [23]:
sv_plants_category["plant_phaseolus_vulgaris|SVgr_3_id_085775|7909"]

{'class': 'Class I (Retrotransposons)',
 'order': 'Non-LTR Retrotransposon',
 'superfamily': 'L1'}

In [24]:
files = []
for i in range(10):
    files.append(f"/Users/nad/NeuralTE/plants_output/domain/{i}.out")

with open("/Users/nad/NeuralTE/plants_output/domain/all.out", "w", encoding="utf-8") as outfile:
    for fname in files:
        with open(fname, "r", encoding="utf-8") as infile:
            outfile.write(infile.read())

In [25]:
neuralte_results = {}

with open("/Users/nad/NeuralTE/plants_output/domain/all.out") as f:
    for line in f:
        parts = line.strip().split("\t")

        name = parts[0]
        type_full = parts[1]

        type_clean = type_full.split("#")[-1]

        neuralte_results[name] = type_clean

print(neuralte_results["plant_oryza_meridionalis|SVgr_11_id_16072|27066"])

LTR/Gypsy


In [26]:
len(set(neuralte_results.values()))

24

In [27]:
all_count = 0
true_count = 0

def has_common_substring(s1, s2, min_len=2):
    for i in range(len(s1) - min_len + 1):
        sub = s1[i:i+min_len]
        if sub in s2:
            return True
    return False


for name in neuralte_results.keys():
    if name not in sv_plants_category:
        continue
    all_count += 1
    # if 'superfamily' not in sv_plants_category[name].keys():
    #     if has_common_substring(neuralte_results[name],
    #                         sv_plants_category[name]['order']):
    #         true_count += 1
    #     else:
    #         print(neuralte_results[name], sv_plants_category[name]['order'])
    if 'superfamily' not in sv_plants_category[name].keys():
        continue
    else:
        if has_common_substring(neuralte_results[name],
                            sv_plants_category[name]['superfamily']):
            true_count += 1
        else:
            print(neuralte_results[name], sv_plants_category[name]['order'])

DNA/hAT-Ac LTR Retrotransposon
LTR/Copia LTR Retrotransposon
LINE/L1 DNA transposon
LTR/Copia LTR Retrotransposon
LTR/Copia DNA transposon
LTR/Gypsy LTR Retrotransposon
LINE/L1 Non-LTR Retrotransposon
DNA/MULE-MuDR DNA transposon
DNA/MULE-MuDR LTR Retrotransposon
LINE/L1 DNA transposon
DNA/PIF-Harbinger LTR Retrotransposon
LINE/L1 DNA transposon
DNA/PIF-Harbinger LTR Retrotransposon
LTR/Gypsy LTR Retrotransposon
LTR/Gypsy DNA transposon
LTR/Caulimovirus DNA transposon
LINE/L1 DNA transposon
LTR/Gypsy LTR Retrotransposon
LTR/Gypsy Non-LTR Retrotransposon
LINE/L1 DNA transposon
DNA/PIF-Harbinger LTR Retrotransposon
DNA/CMC-EnSpm DNA transposon
DNA/CMC-EnSpm LTR Retrotransposon
LTR/Gypsy LTR Retrotransposon
DNA/PIF-Harbinger, LTR Retrotransposon
DNA/PIF-Harbinger LTR Retrotransposon
LTR/Gypsy LTR Retrotransposon
LTR/Copia LTR Retrotransposon
LTR/Copia LTR Retrotransposon
LTR/Copia LTR Retrotransposon
LTR/Gypsy Non-LTR Retrotransposon
DNA/Ginger-1 LTR Retrotransposon
LTR/Copia DNA transpos

In [28]:
true_count / all_count

0.9863599893019523

In [29]:
true_count, all_count

(11064, 11217)

In [30]:
import csv

neuralte_results = {}

with open("/Users/nad/NeuralTE/plants_output/classified.info", newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        neuralte_results[row["#Seq_Name"]] = row["Predict_Label"]

print(neuralte_results['plant_rubus_idaeus|SVgr_6_id_059504|5905'])

KeyError: 'plant_rubus_idaeus|SVgr_6_id_059504|5905'

In [ ]:
all_count = 0
true_count = 0

for name in neuralte_results.keys():
    if name not in sv_plants_category:
        continue
    all_count += 1
    if 'superfamily' not in sv_plants_category[name].keys():
        continue
    else:
        if has_common_substring(neuralte_results[name], sv_plants_category[name]['superfamily']):
            print("same", neuralte_results[name], sv_plants_category[name]['superfamily'])
            true_count += 1
        else:
            print(neuralte_results[name], sv_plants_category[name]['superfamily'])

In [ ]:
print(true_count / all_count)

# Предсказания на кусочках

In [ ]:
import pandas as pd


def load_name_to_class(csv_path: str) -> dict[str, str]:
    df = pd.read_csv(csv_path)

    if "name" not in df.columns or "y_pred" not in df.columns:
        raise ValueError("Ожидаются колонки 'name' и 'y_pred'")

    return dict(zip(df["name"], df["y_pred"]))


# пример
name_to_pred_class = load_name_to_class(
    f"/Users/nad/mobiraph/data/n29_sv_insects_results_30/root/ensemble.csv"
)

In [ ]:
import pandas as pd

HIERARCHY_ROOTS = [
    "root",
    "Class I (Retrotransposons)",
    "Class II (DNA transposons)",
    "Class I (Retrotransposons)\tLTR Retrotransposon",
    "Class I (Retrotransposons)\tNon-LTR Retrotransposon",
]

# загружаем заранее
name_to_pred_superfamily_class2 = load_name_to_class(
    "/Users/nad/mobiraph/data/n29_sv_insects_results_30/Class II (DNA transposons)/ensemble.csv"
)

name_to_pred_order_class1 = load_name_to_class(
    "/Users/nad/mobiraph/data/n29_sv_insects_results_30/Class I (Retrotransposons)/ensemble.csv"
)

name_to_pred_superfamily_ltr = load_name_to_class("/Users/nad/mobiraph/data/n29_sv_insects_results_30/Class I (Retrotransposons)\tLTR Retrotransposon/ensemble.csv")
name_to_pred_superfamily_nonltr = load_name_to_class("/Users/nad/mobiraph/data/n29_sv_insects_results_30/Class I (Retrotransposons)\tNon-LTR Retrotransposon/ensemble.csv")

result = {}

for name, class_name in name_to_pred_class.items():
    if class_name == "Class II (DNA transposons)":
        result[name] = name_to_pred_superfamily_class2.get(name)
    else:
        result[name] = name_to_pred_order_class1.get(name)

for name, class_name in result.items():
    if class_name == 'LTR Retrotransposon':
        result[name] = name_to_pred_superfamily_ltr.get(name)
    elif class_name == 'Non-LTR Retrotransposon':
        result[name] = name_to_pred_superfamily_nonltr.get(name)
    else:
        continue

result

In [ ]:
with open("result.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=4)

In [ ]:
/Users/nad/mobiraph/data/insect_sv_fam_orf_on_repbase_best_30.txt

In [31]:
sv_insects_category_30 = {}

with open("/Users/nad/mobiraph/data/insect_sv_fam_orf_on_repbase_best_30.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if parts[1] not in general_dict:
            continue
        sv_insects_category_30[parts[0]] = general_dict[parts[1]]

In [33]:
sv_insects_category_30

{'name1': {'class': 'Class I (Retrotransposons)',
  'order': 'Non-LTR Retrotransposon',
  'superfamily': 'CR1'},
 'name2': {'class': 'Class II (DNA transposons)',
  'order': 'DNA transposon',
  'superfamily': 'piggyBac'},
 'name3': {'class': 'Class I (Retrotransposons)',
  'order': 'LTR Retrotransposon',
  'superfamily': 'BEL'},
 'name4': {'class': 'Class I (Retrotransposons)',
  'order': 'LTR Retrotransposon',
  'superfamily': 'Gypsy'},
 'name6': {'class': 'Class I (Retrotransposons)',
  'order': 'LTR Retrotransposon',
  'superfamily': 'Gypsy'},
 'name7': {'class': 'Class I (Retrotransposons)',
  'order': 'LTR Retrotransposon',
  'superfamily': 'Gypsy'},
 'name8': {'class': 'Class I (Retrotransposons)',
  'order': 'LTR Retrotransposon',
  'superfamily': 'BEL'},
 'name9': {'class': 'Class I (Retrotransposons)',
  'order': 'Non-LTR Retrotransposon',
  'superfamily': 'Non-LTR Retrotransposon_other'},
 'name10': {'class': 'Class I (Retrotransposons)',
  'order': 'Non-LTR Retrotransposon',

In [34]:
with open("/Users/nad/mobiraph/data/n26_sv_processed/sv_insects_category_30.json", "w", encoding="utf-8") as f:
    json.dump(sv_insects_category_30, f, ensure_ascii=False, indent=4)

In [32]:
HIERARCHY_ROOTS = [
    "root",
    "Class I (Retrotransposons)",
    "Class II (DNA transposons)",
    "Class I (Retrotransposons)\tLTR Retrotransposon",
    "Class I (Retrotransposons)\tNon-LTR Retrotransposon",
]


In [ ]:
sv_insects_category_30_superfamily = {}
for name, info in sv_insects_category_30.items():
    if 'superfamily' not in info:
        continue
    sv_insects_category_30_superfamily[name] = info['superfamily']

In [ ]:
from sklearn.metrics import classification_report

y_true = []
y_pred = []

for name in sv_insects_category_30_superfamily.keys():
    y_true.append(sv_insects_category_30_superfamily[name])
    y_pred.append(result[name])

print(classification_report(y_true, y_pred))

In [ ]:
import csv

neuralte_results_30 = {}

with open("/Users/nad/NeuralTE/insects_output_30/classified.info", newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        neuralte_results_30[row["#Seq_Name"]] = row["Predict_Label"]

print(neuralte_results_30['name1'])

In [38]:
import csv

neuralte_results_plants = {}

with open("/Users/nad/NeuralTE/plants_output/classified.info", newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        neuralte_results_plants[row["#Seq_Name"]] = row["Predict_Label"]

In [ ]:
set(neuralte_results_30.values())

In [ ]:
sv_insects_category_30

In [39]:
superfamily_to_class = {
    # Class 2 — DNA transposons
    "Mariner/Tc1": "Class 2",
    "hAT": "Class 2",
    "MuDR": "Class 2",
    "EnSpm/CACTA": "Class 2",
    "piggyBac": "Class 2",
    "Harbinger": "Class 2",
    "Helitron": "Class 2",
    "Kolobok": "Class 2",
    "Academ": "Class 2",
    "DNA transposon_other": "Class 2",

    # Class 1 — LTR retrotransposons
    "Gypsy": "Class 1",
    "Copia": "Class 1",
    "BEL": "Class 1",
    "DIRS": "Class 1",
    "Troyka": "Class 1",

    # Class 1 — Non-LTR retrotransposons
    "SINE": "Class 1",
    "L1": "Class 1",
    "RTE": "Class 1",
    "CR1": "Class 1",
    "Tx1": "Class 1",
    "RTEX": "Class 1",
    "Tad1": "Class 1",
    "Non-LTR Retrotransposon_other": "Class 1"
}

In [ ]:
set(neuralte_results_30.values())

In [ ]:
superfamily_to_class.keys()

In [40]:
mapping = {
    'Bel-Pao': 'BEL',
    'CACTA': 'EnSpm/CACTA',
    'Copia': 'Copia',
    'Crypton': 'DNA transposon_other',
    'DIRS': 'DIRS',
    'Gypsy': 'Gypsy',
    'Helitron': 'Helitron',
    'I': 'Non-LTR Retrotransposon_other',
    'Jockey': 'Non-LTR Retrotransposon_other',
    'L1': 'L1',
    'Merlin': 'DNA transposon_other',
    'Mutator': 'MuDR',
    'P': 'DNA transposon_other',
    'PIF-Harbinger': 'Harbinger',
    'Penelope': 'Non-LTR Retrotransposon_other',
    'R2': 'Non-LTR Retrotransposon_other',
    'RTE': 'RTE',
    'Retrovirus': 'Gypsy',
    'Tc1-Mariner': 'Mariner/Tc1',
    'Transib': 'DNA transposon_other',
    'Unknown': 'DNA transposon_other',
    'hAT': 'hAT',
    'tRNA': 'SINE'
}

In [42]:
sv_plants_category

{'plant_acer_negundo|SVgr_10_id_08966|7116': {'class': 'Class I (Retrotransposons)',
  'order': 'LTR Retrotransposon',
  'superfamily': 'Gypsy'},
 'plant_acer_negundo|SVgr_10_id_17795|5114': {'class': 'Class I (Retrotransposons)',
  'order': 'LTR Retrotransposon',
  'superfamily': 'Copia'},
 'plant_acer_negundo|SVgr_11_id_23303|6254': {'class': 'Class I (Retrotransposons)',
  'order': 'LTR Retrotransposon',
  'superfamily': 'Copia'},
 'plant_acer_negundo|SVgr_11_id_25259|2321': {'class': 'Class II (DNA transposons)',
  'order': 'DNA transposon',
  'superfamily': 'EnSpm/CACTA'},
 'plant_acer_negundo|SVgr_12_id_21516|5196': {'class': 'Class I (Retrotransposons)',
  'order': 'LTR Retrotransposon',
  'superfamily': 'Copia'},
 'plant_acer_negundo|SVgr_13_id_20896|5170': {'class': 'Class I (Retrotransposons)',
  'order': 'LTR Retrotransposon',
  'superfamily': 'Gypsy'},
 'plant_acer_negundo|SVgr_13_id_26963|6337': {'class': 'Class I (Retrotransposons)',
  'order': 'LTR Retrotransposon',
  's

In [41]:
from sklearn.metrics import classification_report

true_count = 0
all_count = 0

y_true = []
y_pred = []

for name in neuralte_results_plants.keys():
    if name not in sv_plants_category:
        continue

    if 'superfamily' not in sv_plants_category[name]:
        continue

    pred = superfamily_to_class[mapping[neuralte_results_plants[name]]]
    true = superfamily_to_class[sv_plants_category[name]['superfamily']]

    all_count += 1

    y_true.append(true)
    y_pred.append(pred)

    if pred == true:
        true_count += 1

print(true_count / all_count if all_count > 0 else 0)
print(true_count, all_count)

# classification report
print(classification_report(y_true, y_pred))

ValueError: Found empty input array (e.g., `y_true` or `y_pred`) while a minimum of 1 sample is required.

In [ ]:
print(true_count / all_count)

In [ ]:
true_count, all_count